In [2]:
import re
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool

In [3]:
@tool
def customer_lookup(query : str) -> str:
    """Look up customer information"""
    return f"Customer record found for query: {query}"

agent = create_agent(
    model="mistral-small-2506",
    tools=[customer_lookup],
    middleware=[

        # Redact emails
        PIIMiddleware(
            "email",
            strategy="redact", #Redact means replace with [REDACTED]
            apply_to_input=True, #apply to user -> agent before the llm receives it
        ),

        # Mask credit-card-like numbers
        PIIMiddleware(
            pii_type="credit_card",
            detector=r"\b(?:\d{4}[- ]?){3}\d{4}\b",
            strategy="mask",  #Hide only part of the information.
            apply_to_input=True,
        ),

        # Block API keys
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)


In [4]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 7198-3773-9023-2831."
    }]
})

print(result["messages"][0].content)

My email is [REDACTED_EMAIL] and my card is ****-****-****-2831.


In [5]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-2831.', additional_kwargs={}, response_metadata={}, id='1be7052d-9431-4343-8e52-43cf9a07c29d'),
  AIMessage(content="I'm unable to assist with that request. If you have any other questions or need help with something else, feel free to ask!", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 85, 'total_tokens': 113, 'completion_tokens': 28, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019f7961-844d-7c71-9496-e2db6fdde586-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 28, 'total_tokens': 113})]}

In [6]:
#Test API key blocking
try:
    result = agent.invoke({
        "messages": [{
            "role":"user",
            "content": "here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"Blocked as Expected: {e}")

Blocked as Expected: Detected 1 instance(s) of api_key in text content
